# 04 · 排序器資料集

使用完整 OSV‑5M train 的 GPS 建立圖庫、覆蓋率 query、排序器 train query 和 validation query。四組拍攝序列互斥。這本 notebook 比較圖庫大小、鎖定清單並記錄校驗值。

In [3]:
from pathlib import Path
from subprocess import Popen, PIPE, STDOUT, CalledProcessError
from zipfile import ZipFile, ZIP_DEFLATED
import json
import sys

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "scripts/select_ranker_data.py").is_file()), None)
if root is None:
    root = Path("/content/ranker-preprocessing")
    root.mkdir(exist_ok=True)
    with ZipFile("/content/ranker_preprocessing_upload.zip") as archive:
        for member in archive.namelist():
            if not (root / member).resolve().is_relative_to(root.resolve()):
                raise ValueError(member)
        archive.extractall(root)
data = root / "data/ranker_preprocessing"

def run_script(script, *args):
    command = [sys.executable, "-B", "-u", str(root / "scripts" / script), *args]
    with Popen(command, cwd=root, stdout=PIPE, stderr=STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise CalledProcessError(process.returncode, command)
print(root)

/content/ranker-preprocessing


In [4]:
if not (data / "coverage_curve.json").exists():
    run_script("cache_train_metadata.py")
    run_script("select_ranker_data.py")
coverage = json.loads((data / "coverage_curve.json").read_text())
for point in coverage["curve"]:
    print(f"{point['gallery_n']:>7,} | coverage selection {point['coverage']:.2%}"
          f" | random sequences {point['random_sequence']['coverage']:.2%}"
          f" | macro coverage {point['macro_coarse_tile_coverage']:.2%}")
metadata_files = ["coverage_curve.json", "gallery_ordered.jsonl", "query_coverage.jsonl",
                  "query_train.jsonl", "query_validation.jsonl"]
if root == Path("/content/ranker-preprocessing"):
    with ZipFile("/content/ranker-coverage.zip", "w", ZIP_DEFLATED) as archive:
        for name in metadata_files:
            archive.write(data / name, name)
    print("/content/ranker-coverage.zip")

 10,000 | coverage selection 88.12% | random sequences 51.08% | macro coverage 87.71%
 25,000 | coverage selection 97.32% | random sequences 68.08% | macro coverage 95.84%
 50,000 | coverage selection 98.15% | random sequences 79.60% | macro coverage 96.68%
100,000 | coverage selection 98.85% | random sequences 87.87% | macro coverage 97.62%
/content/ranker-coverage.zip


In [5]:
import hashlib
from itertools import islice

# 25k reaches 97.32% GPS coverage. Doubling to 50k adds 0.83 percentage points.
gallery_n = 25_000
source = data / "gallery_ordered.jsonl"
with source.open("rb") as handle:
    selected = b"".join(islice(handle, gallery_n))
gallery_manifest = data / f"gallery_{gallery_n}.jsonl"
if gallery_manifest.exists() and gallery_manifest.read_bytes() != selected:
    raise ValueError("The selected gallery differs from the locked manifest")
if not gallery_manifest.exists():
    partial = gallery_manifest.with_suffix(".part")
    partial.write_bytes(selected)
    partial.replace(gallery_manifest)
files = {"gallery": gallery_manifest, **{s: data / f"query_{s}.jsonl" for s in ("train", "validation", "coverage")}}
expected = {"gallery": gallery_n, "train": 12_000, "validation": 2_000, "coverage": 6_000}
manifest_info = {}
seen_sequences, seen_ids = set(), set()
for split, path in files.items():
    payload = path.read_bytes()
    rows = [json.loads(line) for line in payload.splitlines()]
    ids = {r["id"] for r in rows}
    sequences = {r["sequence"] for r in rows}
    if (len(rows) != expected[split] or len(ids) != len(rows) or len(sequences) != len(rows)
            or ids & seen_ids or sequences & seen_sequences):
        raise ValueError(f"Unexpected count or overlap in {split}")
    seen_ids.update(ids)
    seen_sequences.update(sequences)
    manifest_info[split] = {"file": path.name, "n": len(rows), "sha256": hashlib.sha256(payload).hexdigest()}
    print(f"{split:>10}: {len(rows):,} rows, {len({r['country'] for r in rows})} country/territory codes")
lock = {"osv_revision": coverage["revision"], "manifests": manifest_info,
        "gallery_coverage": next(p for p in coverage["curve"] if p["gallery_n"] == gallery_n)}
partial = data / "dataset.part"
partial.write_text(json.dumps(lock, indent=2) + "\n")
partial.replace(data / "dataset.json")
if root == Path("/content/ranker-preprocessing"):
    with ZipFile("/content/ranker-dataset-25000.zip", "w", ZIP_DEFLATED) as archive:
        for path in (*files.values(), data / "dataset.json", data / "coverage_curve.json"):
            archive.write(path, path.name)
    print("/content/ranker-dataset-25000.zip")

   gallery: 25,000 rows, 215 country/territory codes
     train: 12,000 rows, 199 country/territory codes
validation: 2,000 rows, 178 country/territory codes
  coverage: 6,000 rows, 183 country/territory codes
